In [24]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

from lore_explainer.decision_tree import learn_local_decision_tree

In [2]:
import os

In [3]:
source_file = f'../datasets/german_credit.csv'
class_field = 'default'
# Load and transform dataset
df_g = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [4]:
df_g.head()

,default,account_check_status,duration_in_month,credit_history,purpose,credit_amount,savings,present_emp_since,installment_as_income_perc,personal_status_sex,...,present_res_since,property,age,other_installment_plans,housing,credits_this_bank,job,people_under_maintenance,telephone,foreign_worker
0,0,< 0 DM,6,critical account/ other credits existing (not ...,domestic appliances,1169,unknown/ no savings account,.. >= 7 years,4,male : single,...,4,real estate,67,none,own,2,skilled employee / official,1,"yes, registered under the customers name",yes
1,1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,...,2,real estate,22,none,own,1,skilled employee / official,1,none,yes
2,0,no checking account,12,critical account/ other credits existing (not ...,(vacation - does not exist?),2096,... < 100 DM,4 <= ... < 7 years,2,male : single,...,3,real estate,49,none,own,1,unskilled - resident,2,none,yes
3,0,< 0 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,...,4,if not A121 : building society savings agreeme...,45,none,for free,1,skilled employee / official,2,none,yes
4,1,< 0 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,...,4,unknown / no property,53,none,for free,2,skilled employee / official,2,none,yes


In [5]:
df_g, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df_g, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [6]:
test_size = 0.3
random_state = 42
X_train_g, X_test_g, Y_train_g, Y_test_g= train_test_split(df_g[feature_names], df_g[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df_g[class_field])

In [7]:
bb_g = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb_g.fit(X_train_g.values, Y_train_g.values)
bbox_g = sklearn_classifier_wrapper(bb_g)

## Generate an Explanation

In [8]:
inst_g = X_train_g.iloc[4].values
print('Instance ',inst_g)
print('True class ',Y_train_g.iloc[4])
print('Predicted class ',bb_g.predict(inst_g.reshape(1, -1)))

Instance  [11 7228 1 4 39 2 1 False False False True False True False False False
 False False True False False False False False False False False True
 False False False False False True False False False False False True
 False False True True False False False False True False False True False
 False False False True True False False True]
True class  0
Predicted class  [0]


In [9]:
explainer_g = LoreTabularExplainer(bbox_g)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer_g.fit(df_g, class_field, config)

In [17]:
exp_g=explainer_g.explain(inst_g)

#### Genero il neighbourhood

In [21]:
Z = explainer_g.lore_explainer.neighgen_fn(inst_g,100)

In [22]:
Yb = explainer_g.lore_explainer.bb_predict(Z)
weights =  

In [25]:
idx_train = len(Z) - int(len(Z) * 0.05)
sdt = learn_local_decision_tree(Z[:idx_train], Yb[:idx_train], None, explainer_g.lore_explainer.class_values, False, False, prune_tree=False)

In [27]:
Yc=sdt.predict(Z)

In [28]:
Yc

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0,
       0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 0, 1, 1])

In [30]:
sdt.score(Z,Yb)

0.9666666666666667

In [31]:
Z[0]

array([1.100e+01, 7.228e+03, 1.000e+00, 4.000e+00, 3.900e+01, 2.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 1.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00])

In [33]:
Yc[0]

0

In [40]:
sdt.apply([Z[10]])

array([7])

In [41]:
sdt.decision_path([Z[10]]).indices

array([0, 1, 7], dtype=int32)